# Growth Agent Demo

The Growth Agent is the fifth specialist, now registered alongside Finance, Sentiment, Sales,
and Operations in the boss agent's roster (`backend/agents/boss/registry.py`).

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `regional_sales_breakdown` | Revenue + order count by customer state | Computed directly |
| `market_expansion_signals` | States with strongest revenue growth (recent 6mo vs prior 6mo) | Filters out small-base noise — see below |
| `category_growth_rate` | Category revenue growth, same 6mo/6mo window | Same small-base filter |
| `customer_acquisition_trend` | New customers per month (first-order date, person-level) | Computed directly, trailing-cutoff aware |
| `underperforming_region_diagnosis` | States with declining revenue | Same small-base filter, inverse direction |
| `new_seller_onboarding_rate` | New sellers per month (first-order date) | Computed directly, trailing-cutoff aware |
| `product_gap_analysis` | Revenue-per-seller by category (supply/demand proxy) | **Explicitly NOT true product-gap analysis** — see below |

## Shared infrastructure: `_get_analysis_cutoff()`

Every growth/trend tool here builds on the same trailing-partial-period problem the Sales
Agent found: Olist's data collection stops mid-September 2018. Rather than duplicate the fix
per tool, `_get_analysis_cutoff()` computes it once (same median-based heuristic as Sales) and
every trend tool filters against it before computing anything.

## A second real bug, this time about small-base noise

The first version of `market_expansion_signals()` reported **RR at 827.96% growth** as the
strongest expansion signal. The raw numbers explain why:

| State | Prior revenue | Recent revenue | "Growth" |
|---|---|---|---|
| RR | R$656.61 | R$6,093.09 | 827.96% |
| PI | R$23,898.70 | R$40,317.81 | 68.7% |

RR's prior-period baseline was R$656 — a percentage computed off a near-empty base isn't a
real signal, it's noise. The same thing happened in `category_growth_rate()`: **music** showed
**4849.52% growth** off a R$79.87 prior base. Both tools (plus `underperforming_region_diagnosis`,
which has the identical problem in the opposite direction) now filter out entries below a
minimum prior-period revenue threshold before ranking, matching the `min_reviews`/`min_orders`
guard pattern already used in the Sentiment and Operations agents for the same class of issue.

## `product_gap_analysis` — the tool CLAUDE.md already flagged as weakest-fit

The project spec calls this out directly: *"weakest fit for Olist data; may need scoping down
... flag this when implementing rather than forcing a fake signal."* True product-gap analysis
needs external market/competitor data — visibility into demand for products never listed on
the platform at all. Olist only has transactions that actually happened. What's implemented
instead is a **supply/demand imbalance proxy within existing categories**: revenue-per-seller
as a signal of under- vs. over-supplied categories. The tool's output says this explicitly
rather than presenting it as real whitespace analysis.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.growth import GrowthAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = GrowthAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("Where should we focus growth efforts?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Growth
Execution time: 11899ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.90) Top state by revenue is SP (5163867.22) across 27 states
  source: regional_sales_breakdown

[INFO] (confidence 0.65) Strongest expansion signal is PI at 68.7% growth (recent 6mo vs prior 6mo)
  source: market_expansion_signals

[INFO] (confidence 0.65) Fastest-growing category is construction_tools_lights at 316.38% growth
  source: category_growth_rate

[INFO] (confidence 0.85) New customer acquisition is growing across the last 12 months
  source: customer_acquisition_trend

[WARNING] (confidence 0.65) 8 states show declining revenue; worst is AC at -31.21%
  source: underperforming_region_diagnosis

[INFO] (confidence 0.85) New seller onboarding is growing across the last 12 months
  source: new_seller_onboarding_rate

[INFO] (confidence 0.40) Most undersupplied category (by revenue-per-seller) is computers (24773.68/seller, 9 sellers)
  source: product_gap_analysis



## Verify the small-base noise filter directly

In [4]:
from backend.agents.growth import tools as growth_tools

expansion = growth_tools.market_expansion_signals(db)
print(f"Noise-filtered states: {expansion['noise_filtered_count']} (below R${expansion['min_prior_revenue_threshold']} prior revenue)")
for s in expansion["signals"]:
    print(f"  {s['state']}: {s['growth_pct']}% growth (R${s['prior_revenue']} -> R${s['recent_revenue']})")

Noise-filtered states: 1 (below R$5000 prior revenue)
  PI: 68.7% growth (R$23898.7 -> R$40317.81)
  PR: 44.21% growth (R$206015.73 -> R$297098.21)
  PB: 39.08% growth (R$35061.6 -> R$48762.89)
  TO: 38.67% growth (R$15074.66 -> R$20904.36)
  SP: 32.72% growth (R$1651216.75 -> R$2191518.69)


## Through the boss agent

Five specialists now registered. A query about where growth is coming from, cross-checked
against operational reality, is a natural cross-agent test.

In [5]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("Which regions should we invest in for growth, and can we support that operationally?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Growth', 'Operations']
Overall confidence: 0.75

Synthesis:
Board Memo – Regional Investment & Operational Readiness

**Growth Opportunity**

- Growth's *market_expansion_signals* tool identified PI as the strongest expansion signal with 68.7% growth over the last 6 months (confidence 0.65). This indicates a robust upward trend that merits capital allocation.
- Growth's *regional_sales_breakdown* tool reports SP as the top revenue‑generating state (R$5,163,867.22) across 27 states (confidence 0.9), suggesting continued investment in SP remains justified.
- Growth's *category_growth_rate* tool shows construction_tools_lights growing 316.38% (confidence 0.65). Expanding product mix in PI and SP could capture this high‑growth segment.
- Growth's *new_customer_acquisition_trend* and *new_seller_onboarding_rate* tools both report upward trends over the last 12 months (confidence 0.85), confirming a healthy pipeline of demand and supply.
- Growth's *underperforming_region_d

## What's pending

- Risk, Compliance/HR — 2 specialists remaining
- Once Risk exists, `underperforming_region_diagnosis` (Growth) and Risk's `concentration_risk`/`customer_churn_prediction` are natural cross-references